In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text

url = (
    "https://raw.githubusercontent.com/DataTalksClub/"
    "machine-learning-zoomcamp/main/cohorts/2026/data/"
    "car_fuel_efficiency_2026.csv"
)

# Q1
df = pd.read_csv(url)
df = df.fillna(0)

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
target = "fuel_efficiency_mpg"

y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()
y_test = df_test[target].to_numpy()

train_dicts = df_train.drop(columns=[target]).to_dict(orient="records")
val_dicts = df_val.drop(columns=[target]).to_dict(orient="records")
test_dicts = df_test.drop(columns=[target]).to_dict(orient="records")

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

model = DecisionTreeRegressor(max_depth=1, random_state=1)
model.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
feature_index = model.tree_.feature[0]
split_feature = feature_names[feature_index]

original_feature = split_feature.split("=", 1)[0]

print("Split feature:", split_feature)
print("Original feature:", original_feature)
print()
print(export_text(model, feature_names=list(feature_names)))

Split feature: model_year
Original feature: model_year

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [3]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [4]:
# Q2
rf_q2 = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf_q2.fit(X_train, y_train)

y_pred = rf_q2.predict(X_val)
rmse_q2 = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"Validation RMSE: {rmse_q2:.6f}")
print(f"Answer: {rmse_q2:.3f}")

Validation RMSE: 1.833220
Answer: 1.833


In [5]:
# Q3
results_q3 = []
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    results_q3.append({
        "n_estimators": n,
        "rmse": rmse,
        "rmse_rounded": round(rmse, 3)
    })

results_q3 = pd.DataFrame(results_q3)

print(results_q3.to_string(index=False))

best_index = results_q3["rmse_rounded"].idxmin()
best_n = int(results_q3.loc[best_index, "n_estimators"])

print(f"\nAnswer: {best_n}")

 n_estimators     rmse  rmse_rounded
           10 1.833220         1.833
           50 1.773988         1.774
          100 1.769786         1.770
          150 1.769198         1.769

Answer: 150


In [6]:
# Q4
results_q4 = []
for depth in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        results_q4.append({
            "max_depth": depth,
            "n_estimators": n,
            "rmse": rmse
        })

results_q4 = pd.DataFrame(results_q4)

mean_rmse_q4 = (
    results_q4
    .groupby("max_depth", as_index=False)["rmse"]
    .mean()
    .rename(columns={"rmse": "mean_rmse"})
)

print(mean_rmse_q4.to_string(index=False))

best_index = mean_rmse_q4["mean_rmse"].idxmin()
best_depth = int(mean_rmse_q4.loc[best_index, "max_depth"])

print(f"\nAnswer: {best_depth}")

 max_depth  mean_rmse
        10   1.754392
        15   1.784224
        20   1.786771
        25   1.786305

Answer: 10


In [7]:
# Q5
rf_q5 = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf_q5.fit(X_train, y_train)

feature_importance = pd.Series(
    rf_q5.feature_importances_,
    index=dv.get_feature_names_out(),
    name="importance"
)

options = [
    "vehicle_weight",
    "horsepower",
    "acceleration",
    "engine_displacement"
]

selected_importance = (
    feature_importance
    .loc[options]
    .sort_values(ascending=False)
)

print(selected_importance.to_string())
print(f"\nAnswer: {selected_importance.idxmax()}")

vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998

Answer: vehicle_weight


In [8]:
# Q6
%pip install xgboost


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import xgboost as xgb

feature_names = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    feature_names=feature_names
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    feature_names=feature_names
)

watchlist = [
    (dtrain, "train"),
    (dval, "val")
]

results_q6 = []

for eta in [0.3, 0.1]:
    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1
    }

    print(f"\neta = {eta}")

    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=20
    )

    y_pred = model.predict(dval)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    results_q6.append({
        "eta": eta,
        "rmse": rmse
    })

results_q6 = pd.DataFrame(results_q6)

print("\nSonuçlar:")
print(results_q6.to_string(index=False))

best_index = results_q6["rmse"].idxmin()
best_eta = results_q6.loc[best_index, "eta"]

print(f"\nAnswer: {best_eta}")


eta = 0.3
[0]	train-rmse:2.40653	val-rmse:2.46782
[20]	train-rmse:1.39368	val-rmse:1.73766
[40]	train-rmse:1.25326	val-rmse:1.76296
[60]	train-rmse:1.10265	val-rmse:1.79401
[80]	train-rmse:0.99487	val-rmse:1.81308
[99]	train-rmse:0.90106	val-rmse:1.82646

eta = 0.1
[0]	train-rmse:2.74739	val-rmse:2.78635
[20]	train-rmse:1.55847	val-rmse:1.75076
[40]	train-rmse:1.43074	val-rmse:1.71289
[60]	train-rmse:1.37884	val-rmse:1.71688
[80]	train-rmse:1.34578	val-rmse:1.72100
[99]	train-rmse:1.31192	val-rmse:1.72481

Sonuçlar:
 eta     rmse
 0.3 1.826458
 0.1 1.724810

Answer: 0.1
